In [ ]:
from pyspark.sql import functions as F
from src.analytics.quality_assets import write_quality_assets


dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("quarantine_schema", "quarantine", "Quarantine schema")
dbutils.widgets.text("analytics_schema", "analytics", "Analytics schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
QUARANTINE_SCHEMA = dbutils.widgets.get("quarantine_schema").strip()
ANALYTICS_SCHEMA = dbutils.widgets.get("analytics_schema").strip()

print(
    "Week 3 silver validation namespace: "
    f"silver={CATALOG}.{SILVER_SCHEMA} "
    f"quarantine={CATALOG}.{QUARANTINE_SCHEMA} "
    f"analytics={CATALOG}.{ANALYTICS_SCHEMA}"
)

In [ ]:
required_tables = {
    "silver": ["claims", "providers", "diagnosis", "cost", "policy_chunks"],
    "quarantine": ["claims", "providers", "diagnosis", "cost", "policy_chunks"],
}
missing_tables = []
for table in required_tables["silver"]:
    fqn = f"{CATALOG}.{SILVER_SCHEMA}.{table}"
    try:
        spark.table(fqn).limit(1).collect()
    except Exception as exc:
        missing_tables.append(f"{fqn}: {exc}")
for table in required_tables["quarantine"]:
    fqn = f"{CATALOG}.{QUARANTINE_SCHEMA}.{table}"
    try:
        spark.table(fqn).limit(1).collect()
    except Exception as exc:
        missing_tables.append(f"{fqn}: {exc}")

if missing_tables:
    raise RuntimeError("Missing required Silver/Quarantine tables:
" + "
".join(missing_tables))

trusted_claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claims")
quarantine_claims = spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.claims")
trusted_claim_count = trusted_claims.agg(F.count(F.lit(1)).alias("n")).collect()[0]["n"]
quarantine_claim_count = quarantine_claims.agg(F.count(F.lit(1)).alias("n")).collect()[0]["n"]
if trusted_claim_count <= 0:
    raise AssertionError("Silver claims table is empty; expected trusted claims after validation.")
print(f"Trusted claims: {trusted_claim_count}; quarantined claims: {quarantine_claim_count}")

persisted_assets = write_quality_assets(
    spark,
    catalog=CATALOG,
    silver_schema=SILVER_SCHEMA,
    quarantine_schema=QUARANTINE_SCHEMA,
    analytics_schema=ANALYTICS_SCHEMA,
)

display(
    spark.createDataFrame(
        [{"table_name": table_name, "status": status} for table_name, status in persisted_assets.items()]
    )
)

In [ ]:
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_silver_table_status"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_quarantine_summary"))